# Relative Flux Light Curves using scienceFlux (raw science flux)

## Motivation

In the DIA (Difference Image Analysis) pipeline:

```
psfFlux  =  scienceFlux  -  templateFlux
```

If the template (`templateFlux`) is incorrect — wrong epoch, template noise, residual
gradients, or detector-dependent biases — then `psfFlux` will carry spurious
variations that do **not** originate from the source itself.

This notebook investigates whether the light-curve variability seen in `psfFlux`
for stable Gaia stars is driven by the **science image flux** (`scienceFlux`) or by
a bad **template**.

### Strategy

For each DIA object (top-N by alert count), three sets of 7-panel plots are produced
(one panel per band + one combined panel), with **shared x and y axes** across all
band panels of the same figure:

1. **Section 6** — `scienceFlux / median(scienceFlux)` per band
2. **Section 7** — overlay `scienceFlux/median` (●) vs `psfFlux/median` (□)
3. **Section 8** — estimated template `psfTemplate_est = scienceFlux − psfFlux`,
   normalised to its median

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Created from:** `04_relativeFlux.ipynb`  
- **Last update:** 2026-04-07  
- **Subject:** Fink alert broker — Rubin LSST photometric calibration check via scienceFlux

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_06"
DIR_DATA_IN = "data_FINK_BLOCK_LC_03"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

FILE_BUTLER = os.path.join(DIR_DATA_IN, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_IN, "src_joined_consdb.parquet")

N_TOP = 20  # number of top-ranked DIA objects to plot

BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Fixed column names ────────────────────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"  # difference-image flux
COL_PSFERR = "r:psfFluxErr"

# Candidate names for science-image flux (Fink may vary)
SCIENCE_FLUX_CANDIDATES = [
    "r:scienceFlux",  # most likely after confirmed naming
    "r:psfScience",
    "r:psfScienceFlux",
    "scienceFlux",
    "psfScience",
    "psfScienceFlux",
]
SCIENCE_ERR_CANDIDATES = [
    "r:scienceSigma",
    "r:scienceFluxErr",
    "r:psfScienceSigma",
    "r:psfScienceFluxErr",
    "scienceSigma",
    "scienceFluxErr",
    "psfScienceSigma",
]


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Input directory  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")

## 2. Load data

In [ ]:
if os.path.exists(FILE_BUTLER):
    df = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file: {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file: {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found.\n"
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Source label: {src_label}")

## 3. Schema inspection — detect scienceFlux column

In [ ]:
print("All columns in the loaded table:")
for col in sorted(df.columns):
    print(f"  {col:50s}  dtype={df[col].dtype}")

In [ ]:
COL_SCI = None
COL_SCIERR = None

for candidate in SCIENCE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_SCI = candidate
        print(f"Found science flux column  : '{COL_SCI}'")
        break

for candidate in SCIENCE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_SCIERR = candidate
        print(f"Found science flux err col : '{COL_SCIERR}'")
        break

if COL_SCI is None:
    print()
    print("WARNING: No scienceFlux-like column found in the parquet file.")
    sci_cols = [c for c in df.columns if "science" in c.lower() or "sci" in c.lower()]
    print("Columns containing 'sci' (case-insensitive):")
    print("  " + str(sci_cols) if sci_cols else "  (none)")
else:
    n_valid = df[COL_SCI].notna().sum()
    print(f"Non-NaN values: {n_valid:,} / {len(df):,}  ({100 * n_valid / len(df):.1f}%)")

has_science = COL_SCI is not None
has_sci_err = COL_SCIERR is not None

print(f"\nColumn summary:")
print(f"  psfFlux    (diff)    : {COL_PSF}")
print(f"  scienceFlux          : {COL_SCI}")
print(f"  scienceFluxErr       : {COL_SCIERR}")

## 4. Rank DIA objects by number of alerts (decreasing)

In [ ]:
alert_counts = df.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)
print(f"Total unique DIA objects : {len(alert_counts):,}")
print(f"Top {N_TOP} by alert count:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df[df[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)
print(f"Rows kept for top-{N_TOP} objects : {len(df_top):,}")

## 5. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """Convert MJD array (float) to list of ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def compute_relative_flux(flux, flux_err=None):
    """
    Normalise flux to its median: ratio = flux / median(flux).

    Only finite, positive values are used to compute the median.

    Returns
    -------
    ratio      : ndarray
    ratio_err  : ndarray or None
    f_med      : float  — normalisation median
    sigma_ratio: float  — RMS scatter of ratio over finite+positive values
    """
    f = np.asarray(flux, dtype=float)
    finite_mask = np.isfinite(f) & (f > 0)
    if finite_mask.sum() == 0:
        nan_arr = np.full_like(f, np.nan)
        return nan_arr, nan_arr, np.nan, np.nan
    f_med = float(np.median(f[finite_mask]))
    if f_med == 0.0:
        nan_arr = np.full_like(f, np.nan)
        return nan_arr, nan_arr, 0.0, np.nan
    ratio = f / f_med
    ratio_err = np.asarray(flux_err, dtype=float) / f_med if flux_err is not None else None
    sigma_ratio = float(np.nanstd(ratio[finite_mask]))
    return ratio, ratio_err, f_med, sigma_ratio


def _add_date_axis(ax, dt_array, t0_mjd):
    """Secondary x-axis on top of *ax* showing calendar dates."""
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


def _shared_ylim(ratio_list, margin=0.05):
    """
    Compute a common [ymin, ymax] range from a list of ratio arrays,
    adding a fractional margin around the data extent.
    Returns None if no finite values are found.
    """
    all_vals = np.concatenate([np.asarray(r, dtype=float).ravel() for r in ratio_list])
    finite_vals = all_vals[np.isfinite(all_vals)]
    if len(finite_vals) == 0:
        return None
    vmin, vmax = finite_vals.min(), finite_vals.max()
    span = max(vmax - vmin, 1e-6)
    return vmin - margin * span, vmax + margin * span


def _shared_xlim(dt_list, margin=0.03):
    """
    Compute a common [xmin, xmax] range from a list of Δt arrays.
    """
    all_dt = np.concatenate([np.asarray(d, dtype=float).ravel() for d in dt_list])
    finite_dt = all_dt[np.isfinite(all_dt)]
    if len(finite_dt) == 0:
        return None
    xmin, xmax = finite_dt.min(), finite_dt.max()
    span = max(xmax - xmin, 1e-6)
    return xmin - margin * span, xmax + margin * span


print("Utility functions defined.")

## 6. Per-object plot functions

Three dedicated functions, one per plot type.  Each function:
- creates the 7-panel figure (6 bands + combined),
- enforces **shared x and y scales** across all band panels,
- adds a secondary date axis on top of each panel.

In [ ]:
def plot_science_relflux(obj_id, df_obj, t0_mjd, t0_date, field, src_label, rank):
    """
    Plot scienceFlux / median(scienceFlux) for one DIA object.

    Layout: 7 panels (u g r i z y | all-bands).
    Shared x-axis and shared y-axis across band panels (panel 0-5).
    """
    n_subplots = len(BANDS) + 1
    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        constrained_layout=True,
    )

    # ── Pre-compute all ratios to determine shared axis limits ────────────────
    all_ratios = []
    all_dt = []
    band_data = {}  # band → (dt, ratio, ratio_err, sigma)

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            band_data[band] = None
            continue
        dt = df_b[COL_MJD].values - t0_mjd
        sci_err = df_b[COL_SCIERR].values if has_sci_err else None
        ratio, ratio_err, _, sigma = compute_relative_flux(df_b[COL_SCI].values, sci_err)
        band_data[band] = (dt, ratio, ratio_err, sigma)
        all_ratios.append(ratio)
        all_dt.append(dt)

    ylim = _shared_ylim(all_ratios) if all_ratios else None
    xlim = _shared_xlim(all_dt) if all_dt else None

    # ── Per-band panels ────────────────────────────────────────────────────────
    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        color = BAND_COLORS.get(band, "k")

        if band_data[band] is None:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            if xlim:
                ax.set_xlim(xlim)
            if ylim:
                ax.set_ylim(ylim)
            ax.set_xlabel("Δt [days]", fontsize=8)
            continue

        dt, ratio, ratio_err, sigma = band_data[band]
        ax.errorbar(
            dt,
            ratio,
            yerr=ratio_err,
            fmt="o",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.8,
            capsize=2,
            alpha=0.85,
            label=f"{band}  σ={sigma:.3f}",
        )
        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"band {band}  |  σ={sigma:.3f}", fontsize=9)
        ax.set_ylabel("scienceFlux / median", fontsize=8)
        ax.set_xlabel("Δt [days]", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        _add_date_axis(ax, dt, t0_mjd)

    # ── Combined panel ─────────────────────────────────────────────────────────
    ax_all = axes[-1]
    for band in BANDS:
        if band_data[band] is None:
            continue
        dt, ratio, ratio_err, sigma = band_data[band]
        color = BAND_COLORS.get(band, "k")
        ax_all.errorbar(
            dt,
            ratio,
            yerr=ratio_err,
            fmt="o",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.7,
            capsize=2,
            alpha=0.7,
            label=f"{band} σ={sigma:.3f}",
        )
    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands", fontsize=9)
    ax_all.set_ylabel("scienceFlux / median", fontsize=8)
    ax_all.set_xlabel("Δt [days]", fontsize=8)
    ax_all.legend(fontsize=7, ncol=3, loc="best")
    if xlim:
        ax_all.set_xlim(xlim)
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
        f"  N={len(df_obj)}  |  t₀={t0_date}\n[scienceFlux / median]",
        fontsize=11,
        y=1.06,
    )
    savefig(f"sci_relflux_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()


print("plot_science_relflux() defined.")

In [ ]:
def plot_sci_vs_diff(obj_id, df_obj, t0_mjd, t0_date, field, src_label, rank):
    """
    Overlay scienceFlux/median (●) vs psfFlux/median (□) for one DIA object.

    Shared x-axis across band panels.
    Shared y-axis is computed over ALL series (sci + diff) so both are visible.
    """
    n_subplots = len(BANDS) + 1
    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        constrained_layout=True,
    )

    all_ratios = []
    all_dt = []
    band_data = {}  # band → (dt, sci_ratio, sci_err, sigma_sci, psf_ratio, psf_err, sigma_psf)

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            band_data[band] = None
            continue
        dt = df_b[COL_MJD].values - t0_mjd
        sci_err = df_b[COL_SCIERR].values if has_sci_err else None
        sci_ratio, sci_err_r, _, sigma_sci = compute_relative_flux(df_b[COL_SCI].values, sci_err)
        psf_ratio, psf_err_r, _, sigma_psf = compute_relative_flux(
            df_b[COL_PSF].values, df_b[COL_PSFERR].values
        )
        band_data[band] = (dt, sci_ratio, sci_err_r, sigma_sci, psf_ratio, psf_err_r, sigma_psf)
        all_ratios.extend([sci_ratio, psf_ratio])
        all_dt.append(dt)

    ylim = _shared_ylim(all_ratios) if all_ratios else None
    xlim = _shared_xlim(all_dt) if all_dt else None

    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        color = BAND_COLORS.get(band, "k")

        if band_data[band] is None:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            if xlim:
                ax.set_xlim(xlim)
            if ylim:
                ax.set_ylim(ylim)
            ax.set_xlabel("Δt [days]", fontsize=8)
            continue

        dt, sci_ratio, sci_err_r, sigma_sci, psf_ratio, psf_err_r, sigma_psf = band_data[band]

        ax.errorbar(
            dt,
            sci_ratio,
            yerr=sci_err_r,
            fmt="o",
            ms=5,
            color=color,
            ecolor=color,
            elinewidth=0.9,
            capsize=2,
            alpha=0.90,
            label=f"sci  σ={sigma_sci:.3f}",
        )
        ax.errorbar(
            dt,
            psf_ratio,
            yerr=psf_err_r,
            fmt="s",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.7,
            capsize=2,
            alpha=0.55,
            mfc="none",
            label=f"diff σ={sigma_psf:.3f}",
        )
        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"{band}  sci σ={sigma_sci:.3f} diff σ={sigma_psf:.3f}", fontsize=8)
        ax.set_ylabel("flux / median", fontsize=8)
        ax.set_xlabel("Δt [days]", fontsize=8)
        ax.legend(fontsize=7, loc="best")
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        _add_date_axis(ax, dt, t0_mjd)

    ax_all = axes[-1]
    for band in BANDS:
        if band_data[band] is None:
            continue
        dt, sci_ratio, sci_err_r, _, psf_ratio, psf_err_r, _ = band_data[band]
        color = BAND_COLORS.get(band, "k")
        ax_all.errorbar(
            dt,
            sci_ratio,
            yerr=sci_err_r,
            fmt="o",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.6,
            capsize=2,
            alpha=0.80,
            label=f"{band} sci",
        )
        ax_all.errorbar(
            dt,
            psf_ratio,
            yerr=psf_err_r,
            fmt="s",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.6,
            capsize=2,
            alpha=0.45,
            mfc="none",
            label=f"{band} diff",
        )
    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands — sci (●) vs diff (□)", fontsize=9)
    ax_all.set_ylabel("flux / median", fontsize=8)
    ax_all.set_xlabel("Δt [days]", fontsize=8)
    ax_all.legend(fontsize=6, ncol=4, loc="best")
    if xlim:
        ax_all.set_xlim(xlim)
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
        f"  N={len(df_obj)}  |  t₀={t0_date}\n"
        "scienceFlux/median (●) vs psfFlux/median (□)",
        fontsize=11,
        y=1.07,
    )
    savefig(f"sci_vs_diff_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()


print("plot_sci_vs_diff() defined.")

In [ ]:
def plot_template_relflux(obj_id, df_obj, t0_mjd, t0_date, field, src_label, rank):
    """
    Plot psfTemplate_est / median(psfTemplate_est) for one DIA object,
    where psfTemplate_est = scienceFlux − psfFlux.

    Shared x and y axes across band panels.
    """
    df_obj = df_obj.copy()
    df_obj["psfTemplate_est"] = df_obj[COL_SCI].values - df_obj[COL_PSF].values

    n_subplots = len(BANDS) + 1
    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        constrained_layout=True,
    )

    all_ratios = []
    all_dt = []
    band_data = {}  # band → (dt, ratio, sigma)

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            band_data[band] = None
            continue
        dt = df_b[COL_MJD].values - t0_mjd
        ratio, _, _, sigma = compute_relative_flux(df_b["psfTemplate_est"].values)
        band_data[band] = (dt, ratio, sigma)
        all_ratios.append(ratio)
        all_dt.append(dt)

    ylim = _shared_ylim(all_ratios) if all_ratios else None
    xlim = _shared_xlim(all_dt) if all_dt else None

    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        color = BAND_COLORS.get(band, "k")

        if band_data[band] is None:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            if xlim:
                ax.set_xlim(xlim)
            if ylim:
                ax.set_ylim(ylim)
            ax.set_xlabel("Δt [days]", fontsize=8)
            continue

        dt, ratio, sigma = band_data[band]
        ax.plot(dt, ratio, "D", ms=4, color=color, alpha=0.82, label=f"{band}  σ={sigma:.3f}")
        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(f"band {band}  |  σ={sigma:.3f}", fontsize=9)
        ax.set_ylabel("psfTemplate_est / median", fontsize=8)
        ax.set_xlabel("Δt [days]", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        _add_date_axis(ax, dt, t0_mjd)

    ax_all = axes[-1]
    for band in BANDS:
        if band_data[band] is None:
            continue
        dt, ratio, sigma = band_data[band]
        color = BAND_COLORS.get(band, "k")
        ax_all.plot(dt, ratio, "D", ms=3, color=color, alpha=0.7, label=f"{band} σ={sigma:.3f}")
    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands — estimated template", fontsize=9)
    ax_all.set_ylabel("psfTemplate_est / median", fontsize=8)
    ax_all.set_xlabel("Δt [days]", fontsize=8)
    ax_all.legend(fontsize=7, ncol=3, loc="best")
    if xlim:
        ax_all.set_xlim(xlim)
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
        f"  N={len(df_obj)}  |  t₀={t0_date}\n"
        "Estimated template: psfTemplate_est = scienceFlux − psfFlux  /  median",
        fontsize=11,
        y=1.07,
    )
    savefig(f"template_est_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()


print("plot_template_relflux() defined.")

## 6. Loop: scienceFlux / median — per object

In [ ]:
if not has_science:
    print("scienceFlux column not available — Section 6 skipped.")
else:
    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        t0_mjd = df_obj[COL_MJD].min()
        t0_date = mjd_to_datestr([t0_mjd])[0]
        field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""
        plot_science_relflux(obj_id, df_obj, t0_mjd, t0_date, field, src_label, rank)

## 7. Loop: scienceFlux/median vs psfFlux/median — per object

Key diagnostic: if `psfFlux` (□) varies more than `scienceFlux` (●), the template is the culprit.

In [ ]:
if not has_science:
    print("scienceFlux column not available — Section 7 skipped.")
else:
    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        t0_mjd = df_obj[COL_MJD].min()
        t0_date = mjd_to_datestr([t0_mjd])[0]
        field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""
        plot_sci_vs_diff(obj_id, df_obj, t0_mjd, t0_date, field, src_label, rank)

## 8. Loop: estimated template variability — per object

`psfTemplate_est(t) = scienceFlux(t) − psfFlux(t)`  should be **constant** for a stable template.

In [ ]:
if not has_science:
    print("scienceFlux column not available — Section 8 skipped.")
else:
    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        t0_mjd = df_obj[COL_MJD].min()
        t0_date = mjd_to_datestr([t0_mjd])[0]
        field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""
        plot_template_relflux(obj_id, df_obj, t0_mjd, t0_date, field, src_label, rank)

## 9. Summary table: σ(ratio) — scienceFlux vs psfFlux vs template

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].copy()
    if has_science:
        df_obj["psfTemplate_est"] = df_obj[COL_SCI].values - df_obj[COL_PSF].values
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    row = {"rank": rank + 1, "diaObjectId": obj_id, "n_alerts": len(df_obj), "t0_date": t0_date}

    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"sigma_sci_{band}"] = np.nan
            row[f"sigma_psf_{band}"] = np.nan
            row[f"sigma_tmpl_{band}"] = np.nan
            continue
        _, _, _, sigma_psf = compute_relative_flux(df_b[COL_PSF].values, df_b[COL_PSFERR].values)
        row[f"sigma_psf_{band}"] = round(sigma_psf, 4)
        if has_science:
            sci_err = df_b[COL_SCIERR].values if has_sci_err else None
            _, _, _, sigma_sci = compute_relative_flux(df_b[COL_SCI].values, sci_err)
            _, _, _, sigma_tmpl = compute_relative_flux(df_b["psfTemplate_est"].values)
            row[f"sigma_sci_{band}"] = round(sigma_sci, 4)
            row[f"sigma_tmpl_{band}"] = round(sigma_tmpl, 4)
        else:
            row[f"sigma_sci_{band}"] = np.nan
            row[f"sigma_tmpl_{band}"] = np.nan
    records.append(row)

df_summary = pd.DataFrame(records)
print("σ(relative flux) — sci=scienceFlux  psf=psfFlux(diff)  tmpl=psfTemplate_est")
display(df_summary)

In [ ]:
out_path = os.path.join(DIR_FIGS, f"sigma_summary_{src_label}.csv")
df_summary.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

## 10. Interpretation guide

| Observation | Interpretation |
|---|---|
| σ(sci) ≈ σ(psf) | Template stable; variability is real (atmospheric / gain) |
| σ(psf) > σ(sci) | Template subtracts too much or too little |
| σ(tmpl) >> 0    | Template flux drifts epoch to epoch |
| σ(sci) varies across objects at same epoch | CCD-level flat-field or gain residuals |
| σ(sci) correlated with airmass / seeing | Atmospheric origin (PWV, clouds) |
| σ(sci) correlated across bands, same CCD | Common-mode gain drift |

**Next step notebook:** `07_psfFlux_scienceFlux.ipynb` — direct comparison
of raw `psfFlux` vs `scienceFlux` in nJy (no normalisation).